# Inspect the verified FineWeb cache

This notebook loads the local cache through `load_fineweb_cache`, so the completion marker, manifest, Parquet schemas, part sequence, and row counts are verified before analysis. It displays random-looking but reproducible examples, Qwen token boundaries, token-frequency estimates, and source metadata.

The cache is drawn from [Hugging Face FineWeb `sample-10BT`](https://huggingface.co/datasets/HuggingFaceFW/fineweb). Field meanings and provenance come from the [FineWeb dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb#data-fields). Sampling uses Hugging Face's [streaming buffer shuffle](https://huggingface.co/docs/datasets/stream#shuffle): shards are shuffled, then documents are randomly selected from a rolling buffer and replaced from the source stream. It is deterministic for the fixed seed and is only an approximate global shuffle.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from IPython.display import Markdown, display
from transformers import AutoTokenizer

from ciphers.kirchenbauer_et_al.src.cache_fineweb import load_fineweb_cache

CACHE_NAME = "fineweb-500k"
TOKENIZER_NAME = "Qwen/Qwen3-4B-Base"
PREVIEW_DOCUMENTS = 5
STATISTICS_DOCUMENTS = 10_000
TOKENIZATION_BATCH_SIZE = 64


## Cache manifest and field contract

Every cached row contains: extracted `text`; Common Crawl `id`, `dump`, `url`, crawl `date`, and WARC `file_path`; detected `language` and fastText `language_score`; and `token_count` under the GPT-2 tokenizer. The Qwen token counts computed below are separate.

In [ ]:
cached_dataset = load_fineweb_cache(CACHE_NAME)
cache_directory = Path(os.environ["STEGO_ARTIFACTS_DIR"]) / "datasets" / "fineweb" / CACHE_NAME
manifest = json.loads((cache_directory / "manifest.json").read_text())
display(pd.Series(manifest, name="value").to_frame())

sample_size = max(PREVIEW_DOCUMENTS, STATISTICS_DOCUMENTS)
sampled_documents = list(cached_dataset.take(sample_size))
print(f"Loaded {len(sampled_documents):,} reproducibly shuffled documents from {cache_directory}")


## Example documents and provenance

These are the first examples from the deterministic shuffled stream, not the first sequential documents written to disk. Text is shown in full. URLs point to the pages from which FineWeb extracted the text, though pages may have changed or disappeared since their recorded crawl dates.

In [ ]:
metadata_fields = ["id", "dump", "url", "date", "file_path", "language", "language_score", "token_count"]
for document_index, document in enumerate(sampled_documents[:PREVIEW_DOCUMENTS], start=1):
    display(Markdown(f"### Document {document_index}"))
    display(pd.Series({field: document[field] for field in metadata_fields}, name="value").to_frame())
    print(document["text"])
    print("=" * 100)


## Qwen token boundaries

`token_string` exposes the tokenizer's internal representation; `decoded_piece` shows how the individual token renders. Whitespace or byte-level boundary markers can therefore be distinguished from visible decoded text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
for document_index, document in enumerate(sampled_documents[:PREVIEW_DOCUMENTS], start=1):
    token_ids = tokenizer(document["text"], add_special_tokens=False)["input_ids"]
    token_rows = [
        {
            "position": position,
            "token_id": token_id,
            "token_string": repr(tokenizer.convert_ids_to_tokens(token_id)),
            "decoded_piece": repr(tokenizer.decode([token_id])),
        }
        for position, token_id in enumerate(token_ids)
    ]
    display(Markdown(f"### Document {document_index}: {len(token_ids):,} Qwen tokens"))
    display(pd.DataFrame(token_rows))


## Lengths, source domains, and Common Crawl dumps

These estimates cover only `STATISTICS_DOCUMENTS` documents from the reproducibly shuffled stream. They are not exact statistics for all 500,000 cached documents.

In [ ]:
statistics_documents = sampled_documents[:STATISTICS_DOCUMENTS]
qwen_token_counts = []
token_counter = Counter()
for batch_start in range(0, len(statistics_documents), TOKENIZATION_BATCH_SIZE):
    document_batch = statistics_documents[batch_start : batch_start + TOKENIZATION_BATCH_SIZE]
    encoded_batch = tokenizer(
        [document["text"] for document in document_batch],
        add_special_tokens=False,
        return_attention_mask=False,
        truncation=False,
    )["input_ids"]
    for token_ids in encoded_batch:
        qwen_token_counts.append(len(token_ids))
        token_counter.update(token_ids)

length_statistics = pd.DataFrame(
    {
        "characters": [len(document["text"]) for document in statistics_documents],
        "gpt2_tokens_from_fineweb": [document["token_count"] for document in statistics_documents],
        "qwen_tokens_computed_here": qwen_token_counts,
        "language_score": [document["language_score"] for document in statistics_documents],
    }
)
display(length_statistics.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

source_domains = Counter(urlparse(document["url"]).netloc for document in statistics_documents)
display(pd.DataFrame(source_domains.most_common(25), columns=["source_domain", "documents"]))

common_crawl_dumps = Counter(document["dump"] for document in statistics_documents)
display(pd.DataFrame(common_crawl_dumps.most_common(), columns=["common_crawl_dump", "documents"]))


## 100 most common Qwen tokens

Relative frequency is `token_count / total_Qwen_tokens_in_the_sample`. It is an estimate over the selected documents, not a probability supplied by FineWeb or the model.

In [ ]:
total_qwen_tokens = sum(token_counter.values())
most_common_token_rows = [
    {
        "rank": rank,
        "token_id": token_id,
        "token_string": repr(tokenizer.convert_ids_to_tokens(token_id)),
        "decoded_piece": repr(tokenizer.decode([token_id])),
        "count": count,
        "relative_frequency": count / total_qwen_tokens,
    }
    for rank, (token_id, count) in enumerate(token_counter.most_common(100), start=1)
]
print(f"Counted {total_qwen_tokens:,} Qwen tokens across {len(statistics_documents):,} documents")
display(pd.DataFrame(most_common_token_rows).style.format({"relative_frequency": "{:.6%}"}))
